In [5]:
# Upload and load the dataset and shared fold file

import os
import zipfile
import glob
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Upload both files when prompted:
# 1. Dataset_preparation(1).zip
# 2. fold_assignments.csv
from google.colab import files

uploaded_files = files.upload()

# Extract the project dataset ZIP file
zip_files = [name for name in uploaded_files if name.endswith(".zip")]

if len(zip_files) == 0:
    raise FileNotFoundError("Please upload Dataset_preparation.zip")

zip_path = zip_files[0]
extract_folder = "/content/project_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

# Locate the modeling dataset
dataset_matches = glob.glob(
    os.path.join(extract_folder, "**", "clinvar_expanded_model_features.csv"),
    recursive=True
)

if len(dataset_matches) == 0:
    raise FileNotFoundError("clinvar_expanded_model_features.csv was not found")

dataset_path = dataset_matches[0]

# Locate the shared fold file
fold_matches = glob.glob("/content/fold_assignments.csv")

if len(fold_matches) == 0:
    raise FileNotFoundError("fold_assignments.csv was not uploaded")

fold_path = fold_matches[0]

print("Dataset:", dataset_path)
print("Fold file:", fold_path)

# Read VariationID as string so IDs are not accidentally changed
data = pd.read_csv(dataset_path, dtype={"VariationID": str})
folds = pd.read_csv(fold_path, dtype={"VariationID": str})

print("Dataset shape:", data.shape)
print("Fold file shape:", folds.shape)
print("Dataset columns:")
print(data.columns.tolist())

Saving Dataset_preparation.zip to Dataset_preparation (3).zip
Dataset: /content/project_data/Dataset_preparation/clinvar_expanded_model_features.csv
Fold file: /content/fold_assignments.csv
Dataset shape: (46012, 32)
Fold file shape: (276072, 5)
Dataset columns:
['VariationID', 'GeneSymbol', 'phyloP17way_primate', 'phastCons17way_primate', 'phyloP470way_mammalian', 'phastCons470way_mammalian', 'phyloP100way_vertebrate', 'phastCons100way_vertebrate', 'gnomADe_AF', 'gnomADe_AF_missing', 'gnomADe_AF_neglog10', 'gnomADg_AF', 'gnomADg_AF_missing', 'gnomADg_AF_neglog10', 'gnomAD_AF', 'gnomAD_AF_missing', 'gnomAD_AF_neglog10', 'AA_hydropathy_change', 'AA_hydropathy_abs_change', 'AA_mw_change', 'AA_mw_abs_change', 'AA_volume_change', 'AA_volume_abs_change', 'AA_polarity_change', 'AA_polarity_abs_change', 'AA_charge_change', 'AA_charge_abs_change', 'AA_aromatic_change', 'AA_aromatic_status_changed', 'BLOSUM62_score', 'BinaryClinicalSignificance', 'label']


In [6]:
#Define labels and feature groups

# The target variable:
# 0 = benign or likely benign
# 1 = pathogenic or likely pathogenic
TARGET_COLUMN = "label"

# GeneSymbol is used only for gene-disjoint validation.
# It must not be used as a machine learning feature.
GENE_COLUMN = "GeneSymbol"

# VariationID identifies each genetic variant.
ID_COLUMN = "VariationID"

# These six features describe evolutionary conservation.
# They are divided into three biological groups.
CONSERVATION_GROUPS = {
    "primate": [
        "phyloP17way_primate",
        "phastCons17way_primate"
    ],
    "mammalian": [
        "phyloP470way_mammalian",
        "phastCons470way_mammalian"
    ],
    "vertebrate": [
        "phyloP100way_vertebrate",
        "phastCons100way_vertebrate"
    ]
}

CONSERVATION_FEATURES = [
    feature
    for feature_group in CONSERVATION_GROUPS.values()
    for feature in feature_group
]

# Feature set A is the main feature set for your individual project.
FEATURE_SET_NAME = "A"

# Confirm that all required columns exist.
required_columns = (
    [ID_COLUMN, GENE_COLUMN, TARGET_COLUMN]
    + CONSERVATION_FEATURES
)

missing_columns = [
    column for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Remove rows with missing target or missing gene.
data = data.dropna(subset=[TARGET_COLUMN, GENE_COLUMN]).copy()

# Convert the target to integer form.
data[TARGET_COLUMN] = data[TARGET_COLUMN].astype(int)

# Check the label distribution.
print("Label distribution:")
print(data[TARGET_COLUMN].value_counts())

print("\nNumber of unique variants:", data[ID_COLUMN].nunique())
print("Number of unique genes:", data[GENE_COLUMN].nunique())

Label distribution:
label
0    32013
1    13999
Name: count, dtype: int64

Number of unique variants: 46012
Number of unique genes: 2105


In [8]:
# Verify the shared fold assignments

required_fold_columns = [
    "VariationID",
    "GeneSymbol",
    "validation",
    "repeat",
    "test_fold"
]

missing_fold_columns = [
    column
    for column in required_fold_columns
    if column not in folds.columns
]

if missing_fold_columns:
    raise ValueError(
        f"Missing fold columns: {missing_fold_columns}"
    )

# Keep only the required columns.
folds = folds[required_fold_columns].copy()

# Make sure identifiers have the correct types.
folds["VariationID"] = folds["VariationID"].astype(str)
folds["repeat"] = folds["repeat"].astype(int)
folds["test_fold"] = folds["test_fold"].astype(int)

# Display the available validation modes.
print("Validation modes:")
print(folds["validation"].value_counts())

print("\nRepeats:")
print(sorted(folds["repeat"].unique()))

print("\nTest folds:")
print(sorted(folds["test_fold"].unique()))

# ------------------------------------------------------------
# Important structure of this fold file:
#
# Each variant has:
#   - one standard assignment per repeat
#   - one gene-disjoint assignment per repeat
#
# Therefore, each variant has 3 rows per validation mode,
# not 5 rows.
#
# The five test folds are distributed across all variants.
# ------------------------------------------------------------

# Check that each variant appears exactly once for every
# validation mode and repeat combination.
assignment_counts = (
    folds
    .groupby(["VariationID", "validation", "repeat"])
    .size()
)

if not (assignment_counts == 1).all():
    raise ValueError(
        "Each variant must have exactly one fold assignment "
        "for each validation mode and repeat."
    )

print("\nEach variant has exactly one fold assignment per repeat.")

# Check that every validation mode has all three repeats.
expected_repeats = {0, 1, 2}

for validation_name in folds["validation"].unique():

    actual_repeats = set(
        folds.loc[
            folds["validation"] == validation_name,
            "repeat"
        ].unique()
    )

    if actual_repeats != expected_repeats:
        raise ValueError(
            f"Unexpected repeats for {validation_name}: "
            f"{actual_repeats}"
        )

# Check that every validation mode and repeat uses all five folds.
expected_test_folds = {0, 1, 2, 3, 4}

for validation_name in folds["validation"].unique():

    for repeat_number in sorted(folds["repeat"].unique()):

        actual_test_folds = set(
            folds.loc[
                (folds["validation"] == validation_name)
                & (folds["repeat"] == repeat_number),
                "test_fold"
            ].unique()
        )

        if actual_test_folds != expected_test_folds:
            raise ValueError(
                f"Unexpected test folds for "
                f"{validation_name}, repeat {repeat_number}: "
                f"{actual_test_folds}"
            )

# Show the number of variants assigned to every test fold.
# Standard folds should be approximately equal in size.
# Gene-disjoint folds may have slightly different sizes
# because genes, rather than individual variants, are grouped.
fold_counts = (
    folds
    .groupby(["validation", "repeat", "test_fold"])
    .size()
    .reset_index(name="number_of_variants")
)

print("\nNumber of variants in each test fold:")
display(fold_counts)

print("\nFold assignments are valid.")

Validation modes:
validation
standard         138036
gene_disjoint    138036
Name: count, dtype: int64

Repeats:
[np.int64(0), np.int64(1), np.int64(2)]

Test folds:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Each variant has exactly one fold assignment per repeat.

Number of variants in each test fold:


,validation,repeat,test_fold,number_of_variants
0,gene_disjoint,0,0,9092
1,gene_disjoint,0,1,9684
2,gene_disjoint,0,2,8970
3,gene_disjoint,0,3,9197
4,gene_disjoint,0,4,9069
5,gene_disjoint,1,0,10147
6,gene_disjoint,1,1,9715
7,gene_disjoint,1,2,8891
8,gene_disjoint,1,3,7979
9,gene_disjoint,1,4,9280



Fold assignments are valid.


In [9]:
# Match dataset variants with the fold file

# Make sure the dataset VariationID has the same type.
data[ID_COLUMN] = data[ID_COLUMN].astype(str)

dataset_ids = set(data[ID_COLUMN])
fold_ids = set(folds[ID_COLUMN])

missing_from_folds = dataset_ids - fold_ids
missing_from_dataset = fold_ids - dataset_ids

if missing_from_folds:
    raise ValueError(
        f"{len(missing_from_folds)} dataset variants are missing "
        f"from fold_assignments.csv"
    )

if missing_from_dataset:
    raise ValueError(
        f"{len(missing_from_dataset)} fold variants are missing "
        f"from the modeling dataset"
    )

# Check that every VariationID is unique in the modeling dataset.
if data[ID_COLUMN].duplicated().any():
    raise ValueError(
        "The modeling dataset contains duplicate VariationID values."
    )

# Check that GeneSymbol agrees between the dataset and fold file.
gene_check = folds.merge(
    data[[ID_COLUMN, GENE_COLUMN]],
    on=ID_COLUMN,
    how="left",
    suffixes=("_fold", "_data")
)

gene_mismatches = gene_check[
    gene_check["GeneSymbol_fold"] != gene_check["GeneSymbol_data"]
]

if len(gene_mismatches) > 0:
    raise ValueError(
        "GeneSymbol does not match between the dataset and fold file."
    )

print("Dataset and fold assignments match correctly.")
print("Number of modeling variants:", len(dataset_ids))
print("Number of fold assignments:", len(folds))

Dataset and fold assignments match correctly.
Number of modeling variants: 46012
Number of fold assignments: 276072


In [10]:
# Machine learning settings

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# This is the required shared threshold.
# Every model should use 0.50 for its official pred_label.
DEFAULT_THRESHOLD = 0.50

# Nystroem approximates an RBF kernel.
# Separate Nystroem transformations will be created for:
# 1. Primate features
# 2. Mammalian features
# 3. Vertebrate features
#
# A smaller value makes the experiment faster.
# Increase to 100 or 150 for a final high-quality run if Colab allows.
N_COMPONENTS = 50

# Gamma controls the width of the RBF kernels.
# Because the input data are standardized, 0.5 is a reasonable starting value.
RBF_GAMMA = 0.5

# Regularization values considered during inner training-only tuning.
C_VALUES = [0.1, 1.0, 10.0]

# Candidate weights for combining the three kernels.
# The weights are selected using only the outer training data.
KERNEL_WEIGHT_CANDIDATES = {
    "equal": {
        "primate": 1 / 3,
        "mammalian": 1 / 3,
        "vertebrate": 1 / 3
    },

    "primate_only": {
        "primate": 1.0,
        "mammalian": 0.0,
        "vertebrate": 0.0
    },

    "mammalian_only": {
        "primate": 0.0,
        "mammalian": 1.0,
        "vertebrate": 0.0
    },

    "vertebrate_only": {
        "primate": 0.0,
        "mammalian": 0.0,
        "vertebrate": 1.0
    },

    "primate_mammalian": {
        "primate": 0.5,
        "mammalian": 0.5,
        "vertebrate": 0.0
    },

    "primate_vertebrate": {
        "primate": 0.5,
        "mammalian": 0.0,
        "vertebrate": 0.5
    },

    "mammalian_vertebrate": {
        "primate": 0.0,
        "mammalian": 0.5,
        "vertebrate": 0.5
    }
}

RANDOM_STATE = 42

In [11]:
# Evaluation metrics

def calculate_metrics(y_true, probabilities, threshold=DEFAULT_THRESHOLD):
    """
    Calculate all required classification metrics.

    The official prediction label is created using threshold 0.50.
    A different threshold can be passed later for optional analysis.
    """

    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)

    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    metrics = {
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall_sensitivity": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "mcc": matthews_corrcoef(y_true, predictions),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }

    return metrics

In [12]:
# Use the shared outer folds

# This function does not create a train/test split.
# It reads the exact train/test assignments supplied by the team.

def get_outer_fold_ids(folds, validation_name, repeat_number, fold_number):
    """
    Return the training and testing VariationIDs for one exact
    shared assignment.

    The test set is the selected test_fold.
    The training set is the other four folds from the same
    validation mode and repeat.
    """

    same_repeat = folds[
        (folds["validation"] == validation_name)
        & (folds["repeat"] == repeat_number)
    ].copy()

    test_ids = same_repeat.loc[
        same_repeat["test_fold"] == fold_number,
        "VariationID"
    ].tolist()

    train_ids = same_repeat.loc[
        same_repeat["test_fold"] != fold_number,
        "VariationID"
    ].tolist()

    # Confirm that no variant appears in both sets.
    if set(train_ids).intersection(set(test_ids)):
        raise ValueError("Training and testing variants overlap.")

    return train_ids, test_ids

In [13]:
# Fit preprocessing and separate kernels

def fit_kernel_transformers(training_data):
    """
    Fit preprocessing only on the outer training data.

    This prevents information from the outer test fold leaking into:
    - missing-value imputation
    - scaling
    - kernel approximation
    """

    # Fit one imputer using only training values.
    imputer = SimpleImputer(strategy="median")

    X_train_raw = training_data[CONSERVATION_FEATURES].copy()
    X_train_imputed = imputer.fit_transform(X_train_raw)

    # Fit scaling only on training data.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)

    transformed_training_kernels = {}
    kernel_transformers = {}

    start_index = 0

    for group_name, group_features in CONSERVATION_GROUPS.items():

        number_of_features = len(group_features)
        end_index = start_index + number_of_features

        group_training_data = X_train_scaled[
            :, start_index:end_index
        ]

        # Each biological group receives its own RBF kernel approximation.
        # This is the central idea of the Multi-Kernel SVM.
        n_components_actual = min(
            N_COMPONENTS,
            group_training_data.shape[0]
        )

        nystroem = Nystroem(
            kernel="rbf",
            gamma=RBF_GAMMA,
            n_components=n_components_actual,
            random_state=RANDOM_STATE
        )

        transformed_group = nystroem.fit_transform(
            group_training_data
        )

        transformed_training_kernels[group_name] = transformed_group
        kernel_transformers[group_name] = nystroem

        start_index = end_index

    preprocessing_objects = {
        "imputer": imputer,
        "scaler": scaler,
        "kernel_transformers": kernel_transformers
    }

    return preprocessing_objects, transformed_training_kernels


def transform_data_using_training_objects(data_to_transform, preprocessing_objects):
    """
    Transform validation or testing data using objects already fitted
    on the training data.
    """

    imputer = preprocessing_objects["imputer"]
    scaler = preprocessing_objects["scaler"]
    kernel_transformers = preprocessing_objects["kernel_transformers"]

    X_raw = data_to_transform[CONSERVATION_FEATURES].copy()
    X_imputed = imputer.transform(X_raw)
    X_scaled = scaler.transform(X_imputed)

    transformed_kernels = {}

    start_index = 0

    for group_name, group_features in CONSERVATION_GROUPS.items():

        number_of_features = len(group_features)
        end_index = start_index + number_of_features

        group_data = X_scaled[
            :, start_index:end_index
        ]

        transformed_kernels[group_name] = (
            kernel_transformers[group_name]
            .transform(group_data)
        )

        start_index = end_index

    return transformed_kernels

In [14]:
# Combine the three kernel representations

def combine_kernel_features(transformed_kernels, weights):
    """
    Combine the three separate kernel representations.

    The square root of the weight is used so that the resulting
    feature-space inner product approximates:

        K = wp*Kp + wm*Km + wv*Kv

    This gives each biological evolutionary level its own contribution.
    """

    combined_parts = []

    for group_name in ["primate", "mammalian", "vertebrate"]:
        weight = weights[group_name]

        if weight > 0:
            weighted_features = (
                np.sqrt(weight)
                * transformed_kernels[group_name]
            )

            combined_parts.append(weighted_features)

    if len(combined_parts) == 0:
        raise ValueError("At least one kernel weight must be positive.")

    return np.hstack(combined_parts)

In [15]:
# Tune kernel weights and C using training data only

def get_inner_splits(
    y_training,
    groups_training,
    validation_name,
    random_state
):
    """
    Create inner folds only inside the current outer training set.

    These inner folds are not the final test folds.
    They are used only to choose:
    - the kernel combination
    - the SVM C value

    For gene-disjoint validation, GroupKFold prevents genes from
    appearing in both sides of an inner fold.
    """

    if validation_name == "standard":

        splitter = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=random_state
        )

        return list(
            splitter.split(
                np.zeros(len(y_training)),
                y_training
            )
        )

    elif validation_name == "gene_disjoint":

        splitter = GroupKFold(n_splits=3)

        return list(
            splitter.split(
                np.zeros(len(y_training)),
                y_training,
                groups_training
            )
        )

    else:
        raise ValueError("Unknown validation mode.")


def tune_mkl_parameters(
    transformed_training_kernels,
    y_training,
    groups_training,
    validation_name,
    repeat_number,
    fold_number
):
    """
    Select the kernel weights and SVM C value using inner validation.

    The outer test fold is never used for tuning.
    PR-AUC is used as the tuning objective because the classes
    are not perfectly balanced.
    """

    inner_splits = get_inner_splits(
        y_training=y_training,
        groups_training=groups_training,
        validation_name=validation_name,
        random_state=RANDOM_STATE + repeat_number + fold_number
    )

    tuning_records = []

    best_score = -np.inf
    best_name = None
    best_weights = None
    best_C = None

    for weight_name, weights in KERNEL_WEIGHT_CANDIDATES.items():

        combined_training_features = combine_kernel_features(
            transformed_training_kernels,
            weights
        )

        for C_value in C_VALUES:

            inner_scores = []

            for inner_train_idx, inner_validation_idx in inner_splits:

                model = LinearSVC(
                    C=C_value,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE
                )

                model.fit(
                    combined_training_features[inner_train_idx],
                    y_training[inner_train_idx]
                )

                validation_scores = model.decision_function(
                    combined_training_features[inner_validation_idx]
                )

                score = average_precision_score(
                    y_training[inner_validation_idx],
                    validation_scores
                )

                inner_scores.append(score)

            mean_inner_pr_auc = np.mean(inner_scores)

            tuning_records.append({
                "kernel_weight_name": weight_name,
                "primate_weight": weights["primate"],
                "mammalian_weight": weights["mammalian"],
                "vertebrate_weight": weights["vertebrate"],
                "C": C_value,
                "inner_pr_auc": mean_inner_pr_auc
            })

            if mean_inner_pr_auc > best_score:
                best_score = mean_inner_pr_auc
                best_name = weight_name
                best_weights = weights
                best_C = C_value

    tuning_table = pd.DataFrame(tuning_records)

    return (
        best_name,
        best_weights,
        best_C,
        best_score,
        tuning_table
    )

In [16]:
# Training-only probability calibration

def fit_calibrated_mkl_model(
    combined_training_features,
    y_training,
    groups_training,
    validation_name,
    C_value,
    repeat_number,
    fold_number
):
    """
    Fit the final SVM and a probability calibrator.

    LinearSVC produces decision scores, not probabilities.
    We use out-of-fold training scores to fit a logistic calibrator.

    The calibrator is trained only from the outer training set.
    The outer test set is not used during calibration.
    """

    inner_splits = get_inner_splits(
        y_training=y_training,
        groups_training=groups_training,
        validation_name=validation_name,
        random_state=RANDOM_STATE + 1000 + repeat_number + fold_number
    )

    out_of_fold_scores = np.zeros(len(y_training))

    for inner_train_idx, inner_validation_idx in inner_splits:

        inner_model = LinearSVC(
            C=C_value,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE
        )

        inner_model.fit(
            combined_training_features[inner_train_idx],
            y_training[inner_train_idx]
        )

        out_of_fold_scores[inner_validation_idx] = (
            inner_model.decision_function(
                combined_training_features[inner_validation_idx]
            )
        )

    # Convert decision scores into calibrated pathogenic probabilities.
    calibrator = LogisticRegression(
        max_iter=2000,
        class_weight=None
    )

    calibrator.fit(
        out_of_fold_scores.reshape(-1, 1),
        y_training
    )

    # Fit the final SVM on all outer training examples.
    final_model = LinearSVC(
        C=C_value,
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE
    )

    final_model.fit(
        combined_training_features,
        y_training
    )

    return final_model, calibrator

In [17]:
# STANDARD VALIDATION

def run_mkl_evaluation(validation_name):
    """
    Run Multi-Kernel SVM evaluation using the exact shared folds.

    validation_name must be either:
    - standard
    - gene_disjoint
    """

    all_prediction_rows = []
    all_metric_rows = []
    all_tuning_rows = []

    # Use one unique copy of the modeling data.
    modeling_data = data.drop_duplicates(
        subset=[ID_COLUMN]
    ).set_index(ID_COLUMN)

    for repeat_number in sorted(folds["repeat"].unique()):

        for fold_number in sorted(folds["test_fold"].unique()):

            print(
                f"Running {validation_name} | "
                f"repeat={repeat_number} | fold={fold_number}"
            )

            train_ids, test_ids = get_outer_fold_ids(
                folds=folds,
                validation_name=validation_name,
                repeat_number=repeat_number,
                fold_number=fold_number
            )

            train_data = modeling_data.loc[train_ids].copy()
            test_data = modeling_data.loc[test_ids].copy()

            # Additional safety check for gene-disjoint validation.
            if validation_name == "gene_disjoint":

                training_genes = set(
                    train_data[GENE_COLUMN]
                )

                testing_genes = set(
                    test_data[GENE_COLUMN]
                )

                overlap = training_genes.intersection(testing_genes)

                if len(overlap) > 0:
                    raise ValueError(
                        "Gene overlap detected in gene-disjoint validation."
                    )

            y_train = train_data[TARGET_COLUMN].to_numpy()
            y_test = test_data[TARGET_COLUMN].to_numpy()

            groups_train = train_data[GENE_COLUMN].to_numpy()

            # Fit imputation, scaling, and kernels only on the outer training set.
            preprocessing_objects, transformed_train_kernels = (
                fit_kernel_transformers(train_data)
            )

            # Transform the outer test set using training-fitted objects.
            transformed_test_kernels = (
                transform_data_using_training_objects(
                    test_data,
                    preprocessing_objects
                )
            )

            # Tune using only the outer training set.
            (
                selected_weight_name,
                selected_weights,
                selected_C,
                best_inner_pr_auc,
                tuning_table
            ) = tune_mkl_parameters(
                transformed_training_kernels=transformed_train_kernels,
                y_training=y_train,
                groups_training=groups_train,
                validation_name=validation_name,
                repeat_number=repeat_number,
                fold_number=fold_number
            )

            # Store all inner tuning results for kernel analysis.
            tuning_table["validation"] = validation_name
            tuning_table["repeat"] = repeat_number
            tuning_table["fold"] = fold_number

            all_tuning_rows.append(tuning_table)

            # Combine kernels using the selected weights.
            combined_train = combine_kernel_features(
                transformed_train_kernels,
                selected_weights
            )

            combined_test = combine_kernel_features(
                transformed_test_kernels,
                selected_weights
            )

            # Fit the final SVM and training-only probability calibrator.
            final_model, calibrator = fit_calibrated_mkl_model(
                combined_training_features=combined_train,
                y_training=y_train,
                groups_training=groups_train,
                validation_name=validation_name,
                C_value=selected_C,
                repeat_number=repeat_number,
                fold_number=fold_number
            )

            # Get test decision scores.
            test_decision_scores = final_model.decision_function(
                combined_test
            )

            # Convert decision scores to calibrated probabilities.
            test_probabilities = calibrator.predict_proba(
                test_decision_scores.reshape(-1, 1)
            )[:, 1]

            # Required shared threshold.
            # This remains 0.50 for fair comparison with the other models.
            test_predictions = (
                test_probabilities >= DEFAULT_THRESHOLD
            ).astype(int)

            fold_metrics = calculate_metrics(
                y_true=y_test,
                probabilities=test_probabilities,
                threshold=DEFAULT_THRESHOLD
            )

            fold_metrics.update({
                "model": "Multi-Kernel SVM",
                "feature_set": FEATURE_SET_NAME,
                "validation": validation_name,
                "repeat": repeat_number,
                "fold": fold_number,
                "selected_kernel_weights": selected_weight_name,
                "primate_weight": selected_weights["primate"],
                "mammalian_weight": selected_weights["mammalian"],
                "vertebrate_weight": selected_weights["vertebrate"],
                "selected_C": selected_C,
                "best_inner_pr_auc": best_inner_pr_auc,
                "threshold": DEFAULT_THRESHOLD
            })

            all_metric_rows.append(fold_metrics)

            # Save predictions using the team's shared required format.
            for variant_id, true_label, probability, prediction in zip(
                test_data.index,
                y_test,
                test_probabilities,
                test_predictions
            ):

                all_prediction_rows.append({
                    "model": "Multi-Kernel SVM",
                    "feature_set": FEATURE_SET_NAME,
                    "validation": validation_name,
                    "repeat": repeat_number,
                    "fold": fold_number,
                    "VariationID": variant_id,
                    "label": int(true_label),
                    "p_pathogenic": float(probability),
                    "pred_label": int(prediction)
                })

    predictions_df = pd.DataFrame(all_prediction_rows)
    metrics_df = pd.DataFrame(all_metric_rows)
    tuning_df = pd.concat(
        all_tuning_rows,
        ignore_index=True
    )

    return predictions_df, metrics_df, tuning_df

In [18]:
standard_predictions, standard_metrics, standard_tuning = (
    run_mkl_evaluation("standard")
)

standard_predictions.to_csv(
    "mkl_predictions_standard.csv",
    index=False
)

standard_metrics.to_csv(
    "mkl_metrics_standard.csv",
    index=False
)

standard_tuning.to_csv(
    "mkl_tuning_standard.csv",
    index=False
)

print("Standard validation completed.")
print(standard_predictions.head())
print(standard_metrics.head())

Running standard | repeat=0 | fold=0
Running standard | repeat=0 | fold=1
Running standard | repeat=0 | fold=2
Running standard | repeat=0 | fold=3
Running standard | repeat=0 | fold=4
Running standard | repeat=1 | fold=0
Running standard | repeat=1 | fold=1
Running standard | repeat=1 | fold=2
Running standard | repeat=1 | fold=3
Running standard | repeat=1 | fold=4
Running standard | repeat=2 | fold=0
Running standard | repeat=2 | fold=1
Running standard | repeat=2 | fold=2
Running standard | repeat=2 | fold=3
Running standard | repeat=2 | fold=4
Standard validation completed.
              model feature_set validation  repeat  fold VariationID  label  \
0  Multi-Kernel SVM           A   standard       0     0      309718      0   
1  Multi-Kernel SVM           A   standard       0     0        5045      1   
2  Multi-Kernel SVM           A   standard       0     0      872168      1   
3  Multi-Kernel SVM           A   standard       0     0      813665      1   
4  Multi-Kernel SVM

In [19]:
# GENE-DISJOINT VALIDATION

gene_disjoint_predictions, gene_disjoint_metrics, gene_disjoint_tuning = (
    run_mkl_evaluation("gene_disjoint")
)

gene_disjoint_predictions.to_csv(
    "mkl_predictions_gene_disjoint.csv",
    index=False
)

gene_disjoint_metrics.to_csv(
    "mkl_metrics_gene_disjoint.csv",
    index=False
)

gene_disjoint_tuning.to_csv(
    "mkl_tuning_gene_disjoint.csv",
    index=False
)

print("Gene-disjoint validation completed.")
print(gene_disjoint_predictions.head())
print(gene_disjoint_metrics.head())

Running gene_disjoint | repeat=0 | fold=0
Running gene_disjoint | repeat=0 | fold=1
Running gene_disjoint | repeat=0 | fold=2
Running gene_disjoint | repeat=0 | fold=3
Running gene_disjoint | repeat=0 | fold=4
Running gene_disjoint | repeat=1 | fold=0
Running gene_disjoint | repeat=1 | fold=1
Running gene_disjoint | repeat=1 | fold=2
Running gene_disjoint | repeat=1 | fold=3
Running gene_disjoint | repeat=1 | fold=4
Running gene_disjoint | repeat=2 | fold=0
Running gene_disjoint | repeat=2 | fold=1
Running gene_disjoint | repeat=2 | fold=2
Running gene_disjoint | repeat=2 | fold=3
Running gene_disjoint | repeat=2 | fold=4
Gene-disjoint validation completed.
              model feature_set     validation  repeat  fold VariationID  \
0  Multi-Kernel SVM           A  gene_disjoint       0     0     1008673   
1  Multi-Kernel SVM           A  gene_disjoint       0     0     1083142   
2  Multi-Kernel SVM           A  gene_disjoint       0     0     3129231   
3  Multi-Kernel SVM           

In [20]:
# Combine standard and gene-disjoint predictions

all_mkl_predictions = pd.concat(
    [
        standard_predictions,
        gene_disjoint_predictions
    ],
    ignore_index=True
)

all_mkl_metrics = pd.concat(
    [
        standard_metrics,
        gene_disjoint_metrics
    ],
    ignore_index=True
)

all_mkl_tuning = pd.concat(
    [
        standard_tuning,
        gene_disjoint_tuning
    ],
    ignore_index=True
)

# Confirm the required prediction format.
required_prediction_columns = [
    "model",
    "feature_set",
    "validation",
    "repeat",
    "fold",
    "VariationID",
    "label",
    "p_pathogenic",
    "pred_label"
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in all_mkl_predictions.columns
]

if missing_prediction_columns:
    raise ValueError(
        f"Missing prediction columns: {missing_prediction_columns}"
    )

all_mkl_predictions = all_mkl_predictions[
    required_prediction_columns
]

all_mkl_predictions.to_csv(
    "mkl_predictions_all.csv",
    index=False
)

all_mkl_metrics.to_csv(
    "mkl_metrics_all.csv",
    index=False
)

all_mkl_tuning.to_csv(
    "mkl_tuning_all.csv",
    index=False
)

print("All Multi-Kernel SVM results saved.")
print(all_mkl_predictions.shape)

All Multi-Kernel SVM results saved.
(276072, 9)


In [21]:
# Performance summary

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "mcc",
    "roc_auc",
    "pr_auc"
]

summary_mean = (
    all_mkl_metrics
    .groupby(["model", "feature_set", "validation"])[metric_columns]
    .mean()
    .reset_index()
)

summary_std = (
    all_mkl_metrics
    .groupby(["model", "feature_set", "validation"])[metric_columns]
    .std()
    .reset_index()
)

print("Mean performance:")
display(summary_mean)

print("Standard deviation across folds:")
display(summary_std)

summary_mean.to_csv(
    "mkl_summary_mean.csv",
    index=False
)

summary_std.to_csv(
    "mkl_summary_std.csv",
    index=False
)

Mean performance:


,model,feature_set,validation,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,mcc,roc_auc,pr_auc
0,Multi-Kernel SVM,A,gene_disjoint,0.766844,0.711179,0.630842,0.567215,0.855142,0.59605,0.435121,0.807320,0.609834
1,Multi-Kernel SVM,A,standard,0.769104,0.713914,0.633243,0.572945,0.854882,0.60157,0.440700,0.808643,0.613115


Standard deviation across folds:


,model,feature_set,validation,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,mcc,roc_auc,pr_auc
0,Multi-Kernel SVM,A,gene_disjoint,0.010943,0.010167,0.046884,0.022639,0.013775,0.020640,0.021484,0.009961,0.043444
1,Multi-Kernel SVM,A,standard,0.004851,0.005893,0.008947,0.009408,0.004142,0.008619,0.011876,0.004454,0.007417


In [22]:
# Kernel contribution analysis

# Count how often each kernel-weight configuration was selected.
kernel_selection_counts = (
    all_mkl_metrics
    .groupby(["validation", "selected_kernel_weights"])
    .size()
    .reset_index(name="number_of_outer_folds")
)

print("Selected kernel combinations:")
display(kernel_selection_counts)

# Calculate the average selected weight for each biological group.
average_kernel_weights = (
    all_mkl_metrics
    .groupby("validation")[
        [
            "primate_weight",
            "mammalian_weight",
            "vertebrate_weight"
        ]
    ]
    .mean()
    .reset_index()
)

print("Average selected kernel weights:")
display(average_kernel_weights)

kernel_selection_counts.to_csv(
    "mkl_kernel_selection_counts.csv",
    index=False
)

average_kernel_weights.to_csv(
    "mkl_average_kernel_weights.csv",
    index=False
)

Selected kernel combinations:


,validation,selected_kernel_weights,number_of_outer_folds
0,gene_disjoint,equal,15
1,standard,equal,15


Average selected kernel weights:


,validation,primate_weight,mammalian_weight,vertebrate_weight
0,gene_disjoint,0.333333,0.333333,0.333333
1,standard,0.333333,0.333333,0.333333


In [23]:
# Threshold analysis

def threshold_analysis(predictions_df):
    """
    Compare thresholds for discussion only.

    The official shared pred_label remains based on 0.50.
    This analysis shows whether another threshold changes
    accuracy, MCC, sensitivity, or specificity.
    """

    threshold_values = [
        0.30,
        0.35,
        0.40,
        0.45,
        0.50,
        0.55,
        0.60,
        0.65,
        0.70
    ]

    records = []

    for validation_name in predictions_df["validation"].unique():

        validation_data = predictions_df[
            predictions_df["validation"] == validation_name
        ]

        for threshold in threshold_values:

            metrics = calculate_metrics(
                y_true=validation_data["label"],
                probabilities=validation_data["p_pathogenic"],
                threshold=threshold
            )

            records.append({
                "validation": validation_name,
                "threshold": threshold,
                **metrics
            })

    return pd.DataFrame(records)


threshold_results = threshold_analysis(all_mkl_predictions)

print("Threshold comparison:")
display(threshold_results)

threshold_results.to_csv(
    "mkl_threshold_analysis.csv",
    index=False
)

# The official prediction threshold remains 0.50.
print(
    "Official pred_label threshold used in the shared prediction file:",
    DEFAULT_THRESHOLD
)

Threshold comparison:


,validation,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,mcc,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives
0,standard,0.30,0.735062,0.739789,0.546998,0.751863,0.727715,0.633274,0.447323,0.808562,0.612523,69889,26150,10421,31576
1,standard,0.35,0.750616,0.737762,0.573329,0.704931,0.770593,0.632356,0.452138,0.808562,0.612523,74007,22032,12392,29605
2,standard,0.40,0.760816,0.731908,0.597002,0.658071,0.805746,0.626051,0.451996,0.808562,0.612523,77383,18656,14360,27637
3,standard,0.45,0.766880,0.724321,0.617188,0.615615,0.833026,0.616401,0.448964,0.808562,0.612523,80003,16036,16143,25854
4,standard,0.50,0.769104,0.713914,0.633227,0.572946,0.854882,0.601580,0.440692,0.808562,0.612523,82102,13937,17935,24062
5,standard,0.55,0.764474,0.688479,0.648043,0.494369,0.882589,0.560870,0.410813,0.808562,0.612523,84763,11276,21235,20762
6,standard,0.60,0.758027,0.660672,0.665244,0.412006,0.909339,0.508859,0.378072,0.808562,0.612523,87332,8707,24694,17303
7,standard,0.65,0.737090,0.600345,0.685477,0.251066,0.949625,0.367521,0.293435,0.808562,0.612523,91201,4838,31453,10544
8,standard,0.70,0.717211,0.549045,0.709299,0.119509,0.978582,0.204552,0.204642,0.808562,0.612523,93982,2057,36978,5019
9,gene_disjoint,0.30,0.732403,0.737644,0.543595,0.751030,0.724258,0.630694,0.443006,0.805746,0.604524,69557,26482,10456,31541


Official pred_label threshold used in the shared prediction file: 0.5


In [24]:
# confidence-based abstention analysis

def abstention_analysis(predictions_df):
    """
    Abstention means the model refuses to classify uncertain cases.

    For example:
    - p_pathogenic <= 0.20: predict benign
    - p_pathogenic >= 0.80: predict pathogenic
    - otherwise: abstain

    This does not change the official pred_label column.
    It is an additional uncertainty analysis.
    """

    confidence_ranges = [
        (0.00, 0.50),
        (0.10, 0.90),
        (0.20, 0.80),
        (0.30, 0.70),
        (0.40, 0.60)
    ]

    records = []

    for validation_name in predictions_df["validation"].unique():

        validation_data = predictions_df[
            predictions_df["validation"] == validation_name
        ].copy()

        for lower_probability, upper_probability in confidence_ranges:

            accepted = (
                (validation_data["p_pathogenic"] <= lower_probability)
                |
                (validation_data["p_pathogenic"] >= upper_probability)
            )

            accepted_data = validation_data[accepted].copy()

            if len(accepted_data) == 0:
                continue

            accepted_predictions = (
                accepted_data["p_pathogenic"] >= DEFAULT_THRESHOLD
            ).astype(int)

            accepted_accuracy = accuracy_score(
                accepted_data["label"],
                accepted_predictions
            )

            accepted_mcc = matthews_corrcoef(
                accepted_data["label"],
                accepted_predictions
            )

            coverage = len(accepted_data) / len(validation_data)

            records.append({
                "validation": validation_name,
                "lower_probability": lower_probability,
                "upper_probability": upper_probability,
                "accepted_cases": len(accepted_data),
                "abstained_cases": len(validation_data) - len(accepted_data),
                "coverage": coverage,
                "accuracy_on_accepted_cases": accepted_accuracy,
                "mcc_on_accepted_cases": accepted_mcc
            })

    return pd.DataFrame(records)


abstention_results = abstention_analysis(all_mkl_predictions)

print("Abstention analysis:")
display(abstention_results)

abstention_results.to_csv(
    "mkl_abstention_analysis.csv",
    index=False
)

Abstention analysis:


,validation,lower_probability,upper_probability,accepted_cases,abstained_cases,coverage,accuracy_on_accepted_cases,mcc_on_accepted_cases
0,standard,0.0,0.5,37999,100037,0.275283,0.633227,0.000000
1,standard,0.1,0.9,36303,101733,0.262997,0.948875,0.000000
2,standard,0.2,0.8,63381,74655,0.459163,0.903457,0.056404
3,standard,0.3,0.7,87386,50650,0.633067,0.857208,0.414510
4,standard,0.4,0.6,117753,20283,0.853060,0.804107,0.475974
5,gene_disjoint,0.0,0.5,37754,100282,0.273508,0.630556,0.000000
6,gene_disjoint,0.1,0.9,35640,102396,0.258194,0.948176,0.000000
7,gene_disjoint,0.2,0.8,62816,75220,0.455070,0.903193,0.055336
8,gene_disjoint,0.3,0.7,86582,51454,0.627242,0.856056,0.394274
9,gene_disjoint,0.4,0.6,117203,20833,0.849076,0.803444,0.470827


In [25]:
# CELL 19: Download the important result files

from google.colab import files

output_files = [
    "mkl_predictions_standard.csv",
    "mkl_predictions_gene_disjoint.csv",
    "mkl_predictions_all.csv",
    "mkl_metrics_standard.csv",
    "mkl_metrics_gene_disjoint.csv",
    "mkl_metrics_all.csv",
    "mkl_summary_mean.csv",
    "mkl_summary_std.csv",
    "mkl_tuning_all.csv",
    "mkl_kernel_selection_counts.csv",
    "mkl_average_kernel_weights.csv",
    "mkl_threshold_analysis.csv",
    "mkl_abstention_analysis.csv"
]

for output_file in output_files:
    if os.path.exists(output_file):
        files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (16 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>